In [1]:
# Import required libraries
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import os
from dotenv import load_dotenv
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:
# Database connection configuration to SIMPEG
DB_HOST = os.getenv('DB_HOST_SIMPEG', 'localhost')
DB_PORT = os.getenv('DB_PORT_SIMPEG', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_SIMPEG', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_SIMPEG', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_SIMPEG', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_simpeg = create_engine(connection_string, echo=False)
    with engine_simpeg.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: simpeg_jabar on 10.110.32.121:5432
User: postgres


In [ ]:
# Database connection configuration to TRK 2026
DB_HOST = os.getenv('DB_HOST_2026', 'localhost')
DB_PORT = os.getenv('DB_PORT_2026', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_2026', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_2026', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_2026', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_2026 = create_engine(connection_string, echo=False)
    with engine_2026.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: erk_ekinerja_2026 on 10.110.32.114:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [ ]:
# Define csv, pkl, vpd 
filename = '20260529_proses.csv'
df_vpd_tw1_2026 = pd.read_pickle('df_vpd_tw1_2026.pkl')
tablename = 'v_pegawai_data_tw1_2026'

In [ ]:
# Load the CSV input used for validation
df_validasi_atasan = pd.read_csv(filename)
df_validasi_atasan = df_validasi_atasan.applymap(lambda x: str(x).strip().replace("'", "").replace('"', ""))
df_validasi_atasan['nip_bawahan'] = df_validasi_atasan['nip_bawahan'].str.replace(' ', '')
df_validasi_atasan['nip_atasan'] = df_validasi_atasan['nip_atasan'].str.replace(' ', '')
print(f'Loaded df_validasi_atasan: {df_validasi_atasan.shape}')


Loaded df_validasi_atasan: (118, 2)


In [ ]:
# Load the lookup DataFrame used by the updater
for col in df_vpd_tw1_2026.select_dtypes(include=['float64']).columns:
    df_vpd_tw1_2026[col] = df_vpd_tw1_2026[col].dropna().astype(int).astype(str)
print(f'Loaded df_vpd_tw1_2026: {df_vpd_tw1_2026.shape}')


Loaded df_vpd_tw1_2026: (68172, 53)


In [7]:
def execute_update(query, params, engine, description=""):
    """Execute an UPDATE query and return success status."""
    connection = None
    try:
        connection = engine.connect()
        try:
            connection.rollback()
        except:
            pass
        result = connection.execute(text(query), params)
        rows_affected = result.rowcount
        connection.commit()
        connection.close()
        return {'success': True, 'rows_affected': rows_affected, 'error': None, 'description': description}
    except Exception as e:
        if connection:
            try:
                connection.rollback()
            except:
                pass
            try:
                connection.close()
            except:
                pass
        return {'success': False, 'rows_affected': 0, 'error': str(e), 'description': description}


def atasan_validasi_skp(df_val, df_vpd, target_table, engine):
    """Update atasan validation for one lookup DataFrame and one target table."""
    if df_vpd is None or df_vpd.empty:
        raise ValueError('df_vpd must be provided and non-empty')
    if not isinstance(target_table, str) or not target_table.strip():
        raise ValueError('target_table must be a non-empty string')

    log = []
    for _, row in df_val.iterrows():
        nip_bawahan = str(row.get('nip_bawahan', '')).strip()
        nip_atasan = str(row.get('nip_atasan', '')).strip()
        nama_atasan = None
        peg_id = None

        if 'PLT' in nip_bawahan:
            try:
                jabatan_id = nip_bawahan.split('-')[1] if '-' in nip_bawahan else None
                if jabatan_id:
                    match1 = df_vpd[df_vpd['jabatan_id'] == jabatan_id] if 'jabatan_id' in df_vpd.columns else pd.DataFrame()
                    match2 = df_vpd[df_vpd['tugas_tambahan_jabatan_id'] == jabatan_id] if 'tugas_tambahan_jabatan_id' in df_vpd.columns else pd.DataFrame()
                    matches = pd.concat([match1, match2], ignore_index=True) if (not match1.empty or not match2.empty) else pd.DataFrame()
                    if not matches.empty and 'peg_id' in matches.columns:
                        peg_id = matches['peg_id'].iloc[0]
            except Exception as e:
                log.append({'nip_bawahan': nip_bawahan, 'nip_atasan': nip_atasan, 'table': target_table, 'status': 'failed', 'rows_affected': 0, 'error': f'Error resolving PLT bawahan: {e}'})
                continue

        try:
            if 'PLT' in nip_atasan:
                jabatan_id = nip_atasan.split('-')[1] if '-' in nip_atasan else None
                if jabatan_id:
                    q = 'SELECT jabatan_nama FROM m_spg_jabatan WHERE jabatan_id = :jabatan_id'
                    res = pd.read_sql(text(q), engine, params={'jabatan_id': jabatan_id})
                    if not res.empty:
                        nama_atasan = res['jabatan_nama'].iloc[0]
            else:
                if 'peg_nip' in df_vpd.columns:
                    found = df_vpd[df_vpd['peg_nip'] == nip_atasan]
                    if not found.empty and 'peg_nama' in found.columns:
                        nama_atasan = found['peg_nama'].iloc[0]
        except Exception as e:
            log.append({'nip_bawahan': nip_bawahan, 'nip_atasan': nip_atasan, 'table': target_table, 'status': 'failed', 'rows_affected': 0, 'error': f'Error resolving atasan name: {e}'})
            continue

        try:
            if peg_id is not None and str(peg_id).strip() != '':
                query = f'UPDATE {target_table} SET nip_atasan = :nip_atasan, nama_atasan = :nama_atasan WHERE peg_id = :peg_id'
                params = {'nip_atasan': nip_atasan, 'nama_atasan': nama_atasan, 'peg_id': peg_id}
            else:
                query = f'UPDATE {target_table} SET nip_atasan = :nip_atasan, nama_atasan = :nama_atasan WHERE peg_nip = :nip_bawahan'
                params = {'nip_atasan': nip_atasan, 'nama_atasan': nama_atasan, 'nip_bawahan': nip_bawahan}
            result = execute_update(query, params, engine, f'Update {target_table} for {nip_bawahan}')
            log.append({'nip_bawahan': nip_bawahan, 'nip_atasan': nip_atasan, 'nama_atasan': nama_atasan, 'table': target_table, 'status': 'success' if result['success'] else 'failed', 'rows_affected': result['rows_affected'], 'error': result['error']})
        except Exception as e:
            log.append({'nip_bawahan': nip_bawahan, 'nip_atasan': nip_atasan, 'table': target_table, 'status': 'failed', 'rows_affected': 0, 'error': f'Error executing update: {e}'})

    df_log = pd.DataFrame(log)
    print(f"\n{'='*60}")
    print('UPDATE SUMMARY')
    print(f"{'='*60}")
    print(f"Total operations: {len(df_log)}")
    print(f"Successful: {len(df_log[df_log['status'] == 'success'])}")
    print(f"Failed: {len(df_log[df_log['status'] == 'failed'])}")
    print(f"Total rows affected: {int(df_log['rows_affected'].sum()) if not df_log.empty else 0}")
    print(f"{'='*60}\n")
    if len(df_log[df_log['status'] == 'failed']) > 0:
        print('Failed operations:')
        print(df_log[df_log['status'] == 'failed'][['nip_bawahan', 'table', 'error']])
    return df_log

In [8]:
df_log = atasan_validasi_skp(df_validasi_atasan, df_vpd_tw1_2026, tablename, engine_simpeg)

# Save df_log as csv
now = datetime.now().strftime("%Y%m%d_%H%M%S")
df_log.to_csv(f'df_log_{now}.csv', index=False)


UPDATE SUMMARY
Total operations: 118
Successful: 118
Failed: 0
Total rows affected: 111

